# Audio TTS Preprocessing Pipeline
### Supports: Qwen3-TTS, CosyVoice2, F5-TTS

**Pipeline:**
1. Scan dataset & parse transcripts
2. Analyze audio quality (duration, SNR, clipping, speech ratio)
3. Filter by model-specific criteria
4. Normalize loudness (EBU R128, -23 LUFS)
5. Resample & export in model-specific format

**Dataset Structure:**
- Location: `dataset_dropbox/`
- Speakers: 0011-0020 (10 speakers)
- Emotions: Angry, Happy, Neutral, Sad, Surprise
- Format: WAV 16kHz mono, 16-bit PCM
- Total files: ~17,500

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torchaudio
import torchaudio.transforms as T
import librosa
import soundfile as sf
from pathlib import Path
from tqdm import tqdm

In [ ]:
# Device selection
# MPS (Apple Silicon) for torchaudio ops, CPU fallback otherwise
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    USE_ACCELERATED = True
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    USE_ACCELERATED = True
else:
    DEVICE = torch.device("cpu")
    USE_ACCELERATED = False

print(f"Backend: {'torchaudio' if USE_ACCELERATED else 'librosa'}")
print(f"Device: {DEVICE}")

In [ ]:
DATASET_PATH = Path("/Users/nicholastristan_1/Apple Institute/Challenge 1 Audio/Code/dataset_dropbox")
OUTPUT_PATH = Path("./preprocessed_data")
OUTPUT_PATH.mkdir(exist_ok=True)

TEST_MODE = True
TEST_SPEAKER = '0011'

assert DATASET_PATH.exists(), f"Dataset not found at {DATASET_PATH}"
print(f"Dataset : {DATASET_PATH}")
print(f"Output  : {OUTPUT_PATH}")

In [ ]:
CONFIG = {
    'qwen3_tts': {
        'target_sr': 24000,
        'min_duration': 1.0,
        'max_duration': 15.0,
        'optimal_duration': (10, 15),
        'format': 'wav',
        'bit_depth': 16,
        'channels': 1,
        'output_format': 'jsonl',
        'keep_emotions': None,  # None = keep all
    },
    'cosyvoice2': {
        'target_sr': 16000,  # CosyVoice expects 16kHz prompt audio
        'min_duration': 3.0,
        'max_duration': 10.0,
        'format': 'wav',
        'bit_depth': 16,
        'channels': 1,
        'speech_ratio': 0.6,
        'max_silence': 2.0,
        'output_format': 'jsonl',
        'keep_emotions': None,
    },
    'f5_tts': {
        'target_sr': 24000,
        'min_duration': 2.0,
        'max_duration': 15.0,
        'format': 'wav',
        'bit_depth': 16,
        'channels': 1,
        'output_format': 'jsonl',
        'keep_emotions': None,
    },
}

TARGET_MODEL = 'qwen3_tts'  # Change to: 'qwen3_tts', 'cosyvoice2', or 'f5_tts'
MODEL_CONFIG = CONFIG[TARGET_MODEL]

print(f"Preprocessing for: {TARGET_MODEL.upper()}")
for k, v in MODEL_CONFIG.items():
    print(f"  {k}: {v}")

## 2. Audio Processing Backend

In [ ]:
def load_audio(file_path):
    """Load audio file. Returns (waveform, sample_rate).
    waveform shape: (channels, samples) for torchaudio, (channels, samples) for librosa.
    """
    if USE_ACCELERATED:
        waveform, sr = torchaudio.load(file_path)
        return waveform, sr
    else:
        waveform, sr = librosa.load(file_path, sr=None, mono=False)
        if waveform.ndim == 1:
            waveform = waveform.reshape(1, -1)
        return waveform, sr


def resample_audio(waveform, orig_sr, target_sr):
    if orig_sr == target_sr:
        return waveform
    if USE_ACCELERATED:
        resampler = T.Resample(orig_sr, target_sr)
        return resampler(waveform)
    else:
        return librosa.resample(waveform, orig_sr=orig_sr, target_sr=target_sr)


def to_numpy(waveform):
    """Convert waveform to numpy array regardless of backend."""
    if isinstance(waveform, torch.Tensor):
        return waveform.cpu().numpy()
    return waveform


def save_audio(file_path, waveform, sr):
    if isinstance(waveform, torch.Tensor):
        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)
        torchaudio.save(file_path, waveform.cpu(), sr)
    else:
        if waveform.ndim > 1:
            waveform = waveform[0]
        sf.write(file_path, waveform, sr, subtype='PCM_16')

In [ ]:
def normalize_loudness(audio_np, sr, target_lufs=-23.0):
    """EBU R128 loudness normalization.
    Falls back to peak normalization if pyloudnorm is not installed.

    Ref: EBU R128 standard, https://github.com/csteinmetz1/pyloudnorm
    """
    try:
        import pyloudnorm as pyln
        meter = pyln.Meter(sr)
        current_lufs = meter.integrated_loudness(audio_np)
        if np.isinf(current_lufs):
            # Silent or near-silent audio
            return audio_np
        normalized = pyln.normalize.loudness(audio_np, current_lufs, target_lufs)
        # Prevent clipping
        peak = np.abs(normalized).max()
        if peak > 0.99:
            normalized = normalized * (0.99 / peak)
        return normalized
    except ImportError:
        print("WARNING: pyloudnorm not installed. Using peak normalization.")
        print("Install with: pip install pyloudnorm")
        peak = np.abs(audio_np).max()
        if peak > 0:
            return audio_np * (0.95 / peak)
        return audio_np

## 3. Dataset Parsing

In [ ]:
def parse_transcript_file(txt_path):
    transcripts = {}
    with open(txt_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 3:
                file_id, text, emotion = parts
                transcripts[file_id] = {'text': text, 'emotion': emotion}
    return transcripts


def scan_dataset(dataset_path, test_mode=False, test_speaker=None):
    data = []
    speaker_dirs = sorted([d for d in dataset_path.iterdir()
                           if d.is_dir() and d.name.startswith('00')])

    if test_mode and test_speaker:
        speaker_dirs = [d for d in speaker_dirs if d.name == test_speaker]
        if not speaker_dirs:
            raise ValueError(f"Test speaker {test_speaker} not found!")

    for speaker_dir in tqdm(speaker_dirs, desc="Scanning speakers"):
        speaker_id = speaker_dir.name
        txt_file = speaker_dir / f"{speaker_id}.txt"
        if not txt_file.exists():
            print(f"Warning: No transcript file for {speaker_id}")
            continue

        transcripts = parse_transcript_file(txt_file)
        emotion_dirs = [d for d in speaker_dir.iterdir() if d.is_dir()]

        for emotion_dir in emotion_dirs:
            emotion = emotion_dir.name
            for wav_file in emotion_dir.glob('*.wav'):
                file_id = wav_file.stem
                if file_id in transcripts:
                    data.append({
                        'file_path': str(wav_file),
                        'speaker_id': speaker_id,
                        'emotion': emotion,
                        'file_id': file_id,
                        'transcript': transcripts[file_id]['text'],
                    })

    df = pd.DataFrame(data)
    print(f"Found {len(df)} audio files")
    print(f"Speakers: {df['speaker_id'].nunique()}")
    print(f"Emotions: {df['emotion'].unique().tolist()}")
    return df

In [ ]:
df_metadata = scan_dataset(DATASET_PATH, test_mode=TEST_MODE, test_speaker=TEST_SPEAKER)
df_metadata.head()

In [ ]:
keep_emotions = MODEL_CONFIG.get('keep_emotions', None)

if keep_emotions:
    df_filtered = df_metadata[df_metadata['emotion'].isin(keep_emotions)].copy()
    print(f"Filtered to emotions: {keep_emotions}")
else:
    df_filtered = df_metadata.copy()
    print("Keeping all emotions")

print(f"Files: {len(df_filtered)}")
print(df_filtered['emotion'].value_counts())

## 4. Audio Quality Analysis & Filtering

In [ ]:
def analyze_audio_file(file_path, target_sr):
    """Analyze a single audio file for quality metrics.

    Returns dict with duration, sample_rate, rms, peak_amp, is_clipped,
    speech_ratio, snr_db, valid, and optionally error.
    """
    try:
        waveform, sr = load_audio(file_path)
        audio_np = to_numpy(waveform)

        if audio_np.ndim > 1:
            audio_np = audio_np[0]  # mono

        duration = len(audio_np) / sr
        rms = float(np.sqrt(np.mean(audio_np ** 2)))
        peak_amp = float(np.abs(audio_np).max())
        is_clipped = peak_amp > 0.99

        # Speech ratio: energy-based with adaptive threshold
        # Use RMS of 20ms frames instead of per-sample threshold
        frame_length = int(0.02 * sr)  # 20ms frames
        hop_length = frame_length // 2
        n_frames = max(1, (len(audio_np) - frame_length) // hop_length)
        frame_energies = np.array([
            np.sqrt(np.mean(audio_np[i * hop_length:i * hop_length + frame_length] ** 2))
            for i in range(n_frames)
        ])
        # Adaptive threshold: 10% of mean energy or absolute floor
        energy_threshold = max(np.mean(frame_energies) * 0.1, 1e-4)
        speech_frames = np.sum(frame_energies > energy_threshold)
        speech_ratio = speech_frames / max(n_frames, 1)

        # SNR: use lowest 10% energy frames as noise floor estimate
        # More robust than assuming first 100ms is noise
        sorted_energies = np.sort(frame_energies)
        n_noise_frames = max(1, int(0.1 * len(sorted_energies)))
        noise_floor = np.mean(sorted_energies[:n_noise_frames]) ** 2
        signal_power = np.mean(audio_np ** 2)
        snr_db = float(10 * np.log10(signal_power / (noise_floor + 1e-10)))

        return {
            'duration': duration,
            'sample_rate': int(sr),
            'rms': rms,
            'peak_amp': peak_amp,
            'is_clipped': is_clipped,
            'speech_ratio': speech_ratio,
            'snr_db': snr_db,
            'valid': True,
        }

    except Exception as e:
        return {
            'duration': 0,
            'sample_rate': 0,
            'rms': 0,
            'peak_amp': 0,
            'is_clipped': False,
            'speech_ratio': 0,
            'snr_db': 0,
            'valid': False,
            'error': str(e),
        }

In [ ]:
target_sr = MODEL_CONFIG['target_sr']
analysis_results = []

for idx, row in tqdm(df_filtered.iterrows(), total=len(df_filtered), desc="Analyzing audio"):
    metrics = analyze_audio_file(row['file_path'], target_sr)
    analysis_results.append(metrics)

# Surface errors instead of swallowing them
errors = [r for r in analysis_results if not r['valid']]
if errors:
    print(f"\nFailed files: {len(errors)}/{len(analysis_results)}")
    print("First 5 errors:")
    for e in errors[:5]:
        print(f"  {e.get('error', 'unknown')}")

# Merge into dataframe
for key in analysis_results[0].keys():
    if key != 'error':
        df_filtered[key] = [r[key] for r in analysis_results]

print(f"\nValid: {sum(r['valid'] for r in analysis_results)}")
print(f"Invalid: {len(errors)}")

In [ ]:
valid_df = df_filtered[df_filtered['valid']].copy()

print("AUDIO QUALITY STATISTICS")
print(f"Total analyzed: {len(df_filtered)}")
print(f"Valid: {len(valid_df)}")

if len(valid_df) > 0:
    print(f"\nDuration: {valid_df['duration'].min():.2f}s - {valid_df['duration'].max():.2f}s (mean {valid_df['duration'].mean():.2f}s)")
    print(f"Sample rates: {valid_df['sample_rate'].value_counts().to_dict()}")
    print(f"Speech ratio (mean): {valid_df['speech_ratio'].mean():.2%}")
    print(f"SNR (mean): {valid_df['snr_db'].mean():.1f} dB")
    print(f"Clipped: {valid_df['is_clipped'].sum()}")
else:
    print("\nNo valid files found. Check errors above.")

In [ ]:
min_dur = MODEL_CONFIG['min_duration']
max_dur = MODEL_CONFIG['max_duration']
df_valid = valid_df.copy()

# Duration
df_valid = df_valid[(df_valid['duration'] >= min_dur) & (df_valid['duration'] <= max_dur)]
print(f"After duration filter ({min_dur}s - {max_dur}s): {len(df_valid)}")

# Clipping
df_valid = df_valid[~df_valid['is_clipped']]
print(f"After removing clipped: {len(df_valid)}")

# Speech ratio (if model requires it)
if 'speech_ratio' in MODEL_CONFIG:
    min_speech = MODEL_CONFIG['speech_ratio']
    df_valid = df_valid[df_valid['speech_ratio'] >= min_speech]
    print(f"After speech ratio filter (>={min_speech:.0%}): {len(df_valid)}")

# SNR
min_snr = 5
df_valid = df_valid[df_valid['snr_db'] >= min_snr]
print(f"After SNR filter (>={min_snr}dB): {len(df_valid)}")

print(f"\nFINAL: {len(df_valid)} files ready")
print(f"Filtered out: {len(df_filtered) - len(df_valid)}")

## 5. Preprocess & Export

In [ ]:
def preprocess_and_save_audio(row, output_dir, target_sr):
    """Load, resample, normalize loudness, save."""
    try:
        waveform, sr = load_audio(row['file_path'])

        # Resample
        if sr != target_sr:
            waveform = resample_audio(waveform, sr, target_sr)

        # To numpy for normalization
        audio_np = to_numpy(waveform)
        if audio_np.ndim > 1:
            audio_np = audio_np[0]

        # Loudness normalization (EBU R128, -23 LUFS)
        audio_np = normalize_loudness(audio_np, target_sr, target_lufs=-23.0)

        # Output path: speaker_id/emotion/filename.wav
        speaker_dir = output_dir / row['speaker_id'] / row['emotion']
        speaker_dir.mkdir(parents=True, exist_ok=True)
        output_file = speaker_dir / f"{row['file_id']}.wav"

        sf.write(str(output_file), audio_np, target_sr, subtype='PCM_16')
        return str(output_file)

    except Exception as e:
        print(f"Error processing {row['file_id']}: {e}")
        return None

In [ ]:
output_audio_dir = OUTPUT_PATH / f"{TARGET_MODEL}_audio"
output_audio_dir.mkdir(exist_ok=True)

print(f"Preprocessing {len(df_valid)} files to {MODEL_CONFIG['target_sr']}Hz")
print(f"Output: {output_audio_dir}")

processed_paths = []
for idx, row in tqdm(df_valid.iterrows(), total=len(df_valid), desc="Processing"):
    output_path = preprocess_and_save_audio(row, output_audio_dir, MODEL_CONFIG['target_sr'])
    processed_paths.append(output_path)

df_valid['processed_path'] = processed_paths
df_valid = df_valid[df_valid['processed_path'].notna()].copy()
print(f"Processed: {len(df_valid)} files")

## 6. Model-Specific Output Format

In [ ]:
if TARGET_MODEL == 'qwen3_tts':
    for speaker_id, group in df_valid.groupby('speaker_id'):
        output_file = OUTPUT_PATH / f"{speaker_id}_{TARGET_MODEL}.jsonl"
        ref_audio = group['processed_path'].iloc[0]

        with open(output_file, 'w', encoding='utf-8') as f:
            for _, row in group.iterrows():
                data = {
                    "audio": row['processed_path'],
                    "text": row['transcript'],
                    "ref_audio": ref_audio,
                }
                f.write(json.dumps(data, ensure_ascii=False) + '\n')
        print(f"  {output_file} ({len(group)} samples)")

elif TARGET_MODEL == 'cosyvoice2':
    for speaker_id, group in df_valid.groupby('speaker_id'):
        output_file = OUTPUT_PATH / f"{speaker_id}_{TARGET_MODEL}.jsonl"
        with open(output_file, 'w', encoding='utf-8') as f:
            for _, row in group.iterrows():
                data = {
                    "audio_path": row['processed_path'],
                    "text": row['transcript'],
                    "speaker": speaker_id,
                    "emotion": row['emotion'],
                }
                f.write(json.dumps(data, ensure_ascii=False) + '\n')
        print(f"  {output_file} ({len(group)} samples)")

elif TARGET_MODEL == 'f5_tts':
    # F5-TTS uses reference audio + text pairs
    # Ref: https://github.com/SWivid/F5-TTS
    for speaker_id, group in df_valid.groupby('speaker_id'):
        output_file = OUTPUT_PATH / f"{speaker_id}_{TARGET_MODEL}.jsonl"
        with open(output_file, 'w', encoding='utf-8') as f:
            for _, row in group.iterrows():
                data = {
                    "audio_path": row['processed_path'],
                    "text": row['transcript'],
                    "speaker": speaker_id,
                }
                f.write(json.dumps(data, ensure_ascii=False) + '\n')
        print(f"  {output_file} ({len(group)} samples)")

In [ ]:
metadata_file = OUTPUT_PATH / f"{TARGET_MODEL}_metadata.csv"
df_valid.to_csv(metadata_file, index=False)
print(f"Saved metadata: {metadata_file}")

## 7. Analysis

In [ ]:
print("FEATURE ANALYSIS REPORT")
print(f"Model: {TARGET_MODEL.upper()}")
print(f"Total processed: {len(df_valid)}")

if len(df_valid) > 0:
    print("\n--- Per-Speaker ---")
    speaker_stats = df_valid.groupby('speaker_id').agg({
        'duration': ['count', 'sum', 'mean'],
        'snr_db': 'mean',
        'speech_ratio': 'mean'
    }).round(2)
    print(speaker_stats)

    print("\n--- Per-Emotion ---")
    emotion_stats = df_valid.groupby('emotion').agg({
        'duration': ['count', 'mean'],
        'snr_db': 'mean'
    }).round(2)
    print(emotion_stats)

    print(f"\nTotal audio: {df_valid['duration'].sum() / 3600:.2f}h")
    print(f"Mean SNR: {df_valid['snr_db'].mean():.1f} dB")
    print(f"Mean speech ratio: {df_valid['speech_ratio'].mean():.2%}")

In [ ]:
try:
    import matplotlib.pyplot as plt

    if len(df_valid) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))

        axes[0, 0].hist(df_valid['duration'], bins=50, edgecolor='black')
        axes[0, 0].set_xlabel('Duration (s)')
        axes[0, 0].set_ylabel('Count')
        axes[0, 0].set_title('Duration Distribution')
        axes[0, 0].axvline(MODEL_CONFIG['min_duration'], color='r', linestyle='--', label='Min')
        axes[0, 0].axvline(MODEL_CONFIG['max_duration'], color='r', linestyle='--', label='Max')
        axes[0, 0].legend()

        axes[0, 1].hist(df_valid['snr_db'], bins=50, edgecolor='black')
        axes[0, 1].set_xlabel('SNR (dB)')
        axes[0, 1].set_ylabel('Count')
        axes[0, 1].set_title('SNR Distribution')

        speaker_counts = df_valid['speaker_id'].value_counts().sort_index()
        axes[1, 0].bar(speaker_counts.index, speaker_counts.values)
        axes[1, 0].set_xlabel('Speaker ID')
        axes[1, 0].set_ylabel('Count')
        axes[1, 0].set_title('Files per Speaker')
        axes[1, 0].tick_params(axis='x', rotation=45)

        emotion_counts = df_valid['emotion'].value_counts()
        axes[1, 1].bar(emotion_counts.index, emotion_counts.values)
        axes[1, 1].set_xlabel('Emotion')
        axes[1, 1].set_ylabel('Count')
        axes[1, 1].set_title('Files per Emotion')
        axes[1, 1].tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plot_file = OUTPUT_PATH / f"{TARGET_MODEL}_analysis.png"
        plt.savefig(plot_file, dpi=150)
        print(f"Saved: {plot_file}")
        plt.show()

except ImportError:
    print("matplotlib not available, skipping plots")

In [ ]:
print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"Model: {TARGET_MODEL.upper()}")
print(f"Output: {OUTPUT_PATH}")
print(f"Files: {len(df_valid)}")
if len(df_valid) > 0:
    print(f"Audio: {df_valid['duration'].sum() / 3600:.2f}h")
print(f"Format: {MODEL_CONFIG['target_sr']}Hz, mono, 16-bit, loudness-normalized (-23 LUFS)")

print("\nOutput files:")
for f in sorted(OUTPUT_PATH.glob(f"*{TARGET_MODEL}*")):
    print(f"  {f.name}")